In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score,root_mean_squared_error,mean_absolute_error
import os
from tqdm import tqdm

In [8]:
os.chdir('D:\Machine Learning\Datasets')
df = pd.read_csv("boston.csv")
df

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,black,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,0.06263,0.0,11.93,0,0.573,6.593,69.1,2.4786,1,273,21.0,391.99,9.67,22.4
502,0.04527,0.0,11.93,0,0.573,6.120,76.7,2.2875,1,273,21.0,396.90,9.08,20.6
503,0.06076,0.0,11.93,0,0.573,6.976,91.0,2.1675,1,273,21.0,396.90,5.64,23.9
504,0.10959,0.0,11.93,0,0.573,6.794,89.3,2.3889,1,273,21.0,393.45,6.48,22.0


In [9]:
from sklearn.compose import make_column_selector
ohe=OneHotEncoder(drop='first',sparse_output=False).set_output(transform='pandas') # one hot encoder
col_trnf=ColumnTransformer([('OHE',ohe,make_column_selector(dtype_include=object))],remainder='passthrough',verbose_feature_names_out=True)
col_trnf=col_trnf.set_output(transform='pandas')
X=col_trnf.fit_transform(df)

In [10]:
X=df.drop('medv',axis=1)
y=df['medv']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Hyper Parameter tuning

In [11]:
alphas = np.linspace(.001,15,30)
l1 = np.linspace(0,1,10)

In [12]:

scores=[]
for a in alphas:
    for r in l1:
        el=ElasticNet(alpha=a)
        el.fit(X_train,y_train)
        y_pred=el.predict(X_test)
        scores.append([a,r,mean_absolute_error(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=['alpha','score','l1_ratio'])
df_scores.sort_values('score')

,alpha,score,l1_ratio
0,0.001000,0.0,3.163190
30,1.552621,0.0,3.773290
40,2.069828,0.0,3.883949
270,13.965586,0.0,4.508732
50,2.587034,0.0,3.981261
...,...,...,...
39,1.552621,1.0,3.773290
29,1.035414,1.0,3.647275
269,13.448379,1.0,4.504140
49,2.069828,1.0,3.883949


# Housing Data

In [13]:
housing=pd.read_csv('Housing.csv')

In [14]:
# column transformer
cat_hous=housing[housing.columns[housing.dtypes==object]] # seperating catogorical data
num_hous=housing[housing.columns[housing.dtypes!=object]]
ohe=OneHotEncoder(drop='first',sparse_output=False).set_output(transform='pandas') # one hot encoder
ohe_house=ohe.fit_transform(cat_hous) # cat to 1,0
house=pd.concat([num_hous,ohe_house],axis=1) # row-wise concat
X,y=house.drop('price',axis=1),house['price']


In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [16]:

scores=[]
for a in alphas:
    for r in l1:
        el=ElasticNet(alpha=a)
        el.fit(X_train,y_train)
        y_pred=el.predict(X_test)
        scores.append([a,r,mean_absolute_error(y_test,y_pred)])
df_scores=pd.DataFrame(scores,columns=['alpha','score','l1_ratio'])
df_scores.sort_values('score')

,alpha,score,l1_ratio
0,0.001000,0.0,11503.509012
30,1.552621,0.0,14395.075737
40,2.069828,0.0,14877.310106
270,13.965586,0.0,17552.810054
50,2.587034,0.0,15249.545655
...,...,...,...
39,1.552621,1.0,14395.075737
29,1.035414,1.0,13727.623670
269,13.448379,1.0,17521.678949
49,2.069828,1.0,14877.310106
